# Neural Networks Notebook

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks

# Set random seeds
np.random.seed(42)
tf.random.set_seed(42)

# Load  dataset
df = pd.read_csv('energydata_complete.csv')

print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")

## 2. Exploratory Data Analysis & Preprocessing

In [ ]:
print(f"Dataset shape: {df.shape}")
print(df.info())
print(df.describe())

# Convert date and extract time features
df['date'] = pd.to_datetime(df['date'])
df['hour'] = df['date'].dt.hour
df['day_of_week'] = df['date'].dt.dayofweek
df['month'] = df['date'].dt.month

# Feature correlation analysis
correlation_matrix = df.drop(columns=['date', 'rv1', 'rv2']).corr()
target_correlation = correlation_matrix['Appliances'].sort_values(ascending=False)

print("\nTop 10 features correlated with Appliances:")
print(target_correlation.head(10))

# Define features (X) and target (y)
X = df.drop(columns=['Appliances', 'date', 'rv1', 'rv2'])
y = df['Appliances']

# Train-test split (use shuffle=False if respecting time series order)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True
)

# Standardise features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nTraining samples: {X_train_scaled.shape[0]}, Features: {X_train_scaled.shape[1]}")
print(f"Target range: {y.min():.2f} to {y.max():.2f} Wh")
print(f"Target mean: {y.mean():.2f} Wh, std: {y.std():.2f} Wh")

## 3. Baseline Neural Network Model

In [ ]:
def create_baseline_model(input_dim):
    model = models.Sequential([
        layers.Dense(64, activation='relu', input_shape=(input_dim,)),
        layers.Dense(32, activation='relu'),
        layers.Dense(1)  # Output layer for regression
    ])
    model.compile(
        optimizer='adam',
        loss='mse',
        metrics=['mae']
    )
    return model

baseline_model = create_baseline_model(X_train_scaled.shape[1])
baseline_model.summary()

# Train the baseline model
print("\nTraining baseline model...")
history_baseline = baseline_model.fit(
    X_train_scaled, y_train,
    validation_split=0.15,
    epochs=50,
    batch_size=32,
    verbose=1,
    callbacks=[callbacks.EarlyStopping(patience=5, restore_best_weights=True)]
)

# Evaluate baseline
y_pred_baseline = baseline_model.predict(X_test_scaled).flatten()
test_rmse_baseline = np.sqrt(mean_squared_error(y_test, y_pred_baseline))
test_r2_baseline = r2_score(y_test, y_pred_baseline)
test_mae_baseline = np.mean(np.abs(y_test - y_pred_baseline))
test_mape_baseline = np.mean(np.abs((y_test - y_pred_baseline) / y_test)) * 100

print(f"\n--- Baseline Model Results ---")
print(f"Test RMSE: {test_rmse_baseline:.2f} Wh")
print(f"Test MAE: {test_mae_baseline:.2f} Wh")
print(f"Test MAPE: {test_mape_baseline:.2f}%")
print(f"Test R²: {test_r2_baseline:.3f}")
print(f"MAE/Mean Ratio: {test_mae_baseline/y_test.mean():.2%}")

## 4. Hyperparameter Tuning & Model Improvement

In [ ]:
def build_model(hidden_layers, neurons_per_layer, activation='relu', dropout_rate=0.2):
    model = models.Sequential()
    model.add(layers.Input(shape=(X_train_scaled.shape[1],)))
    
    for i in range(hidden_layers):
        model.add(layers.Dense(neurons_per_layer, activation=activation))
        if i < hidden_layers - 1:  # No dropout on last hidden layer
            model.add(layers.Dropout(dropout_rate))
    
    model.add(layers.Dense(1))
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

# Enhanced configurations to test
configs = [
    {'hidden_layers': 2, 'neurons': 64, 'dropout': 0.1},
    {'hidden_layers': 3, 'neurons': 128, 'dropout': 0.2},
    {'hidden_layers': 4, 'neurons': 64, 'dropout': 0.2},
    {'hidden_layers': 3, 'neurons': 256, 'dropout': 0.3},
    {'hidden_layers': 2, 'neurons': 128, 'dropout': 0.1}
]

tuning_results = []
histories = []

for i, config in enumerate(configs):
    print(f"\nTesting config {i+1}/{len(configs)}: {config}")
    model = build_model(config['hidden_layers'], config['neurons'], dropout_rate=config['dropout'])
    
    history = model.fit(
        X_train_scaled, y_train,
        validation_split=0.15,
        epochs=30,
        batch_size=32,
        verbose=0,
        callbacks=[callbacks.EarlyStopping(patience=5, restore_best_weights=True)]
    )
    
    y_pred = model.predict(X_test_scaled, verbose=0).flatten()
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    tuning_results.append({**config, 'test_rmse': rmse, 'test_r2': r2})
    histories.append(history)
    
    print(f"  RMSE: {rmse:.2f} Wh, R²: {r2:.3f}")

# Display tuning results
results_df = pd.DataFrame(tuning_results)
print("\n--- Hyperparameter Tuning Results ---")
print(results_df.sort_values('test_rmse'))

# Find best configuration
best_config = results_df.loc[results_df['test_rmse'].idxmin()]
print(f"\nBest configuration: {best_config.to_dict()}")

## 5. Final Model & Evaluation

In [ ]:
# --- 5. FINAL MODEL & EVALUATION ---
# Based on tuning results, define improved final architecture
final_model = models.Sequential([
    layers.Dense(256, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    layers.Dropout(0.2),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(1)
])

# Enhanced optimizer with weight decay
optimizer = keras.optimizers.Adam(
    learning_rate=0.001,
    beta_1=0.9,
    beta_2=0.999,
    weight_decay=0.0001
)

final_model.compile(
    optimizer=optimizer,
    loss='huber',  # More robust to outliers than MSE
    metrics=['mae', 'mse']
)

final_model.summary()

print("\nTraining final model...")
history_final = final_model.fit(
    X_train_scaled, y_train,
    validation_split=0.15,
    epochs=100,
    batch_size=32,
    verbose=1,
    callbacks=[
        callbacks.EarlyStopping(
            monitor='val_loss', 
            patience=15,  # Increased patience
            restore_best_weights=True,
            min_delta=0.001
        ),
        callbacks.ReduceLROnPlateau(
            monitor='val_loss', 
            factor=0.5, 
            patience=7,  # Changed from 5
            min_lr=0.00001
        )
    ]
)

# Final evaluation
y_train_pred_final = final_model.predict(X_train_scaled).flatten()
y_test_pred_final = final_model.predict(X_test_scaled).flatten()

# Calculate comprehensive metrics
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

train_rmse_final = np.sqrt(mean_squared_error(y_train, y_train_pred_final))
test_rmse_final = np.sqrt(mean_squared_error(y_test, y_test_pred_final))
train_mae_final = mean_absolute_error(y_train, y_train_pred_final)
test_mae_final = mean_absolute_error(y_test, y_test_pred_final)
train_mape_final = mean_absolute_percentage_error(y_train, y_train_pred_final) * 100
test_mape_final = mean_absolute_percentage_error(y_test, y_test_pred_final) * 100
train_r2_final = r2_score(y_train, y_train_pred_final)
test_r2_final = r2_score(y_test, y_test_pred_final)

print("\n" + "="*60)
print("FINAL MODEL PERFORMANCE")
print("="*60)
print(f"Training RMSE: {train_rmse_final:.2f} Wh | R²: {train_r2_final:.3f}")
print(f"Test RMSE:     {test_rmse_final:.2f} Wh | R²: {test_r2_final:.3f}")
print(f"Test MAE:      {test_mae_final:.2f} Wh | MAPE: {test_mape_final:.2f}%")
print(f"MAE/Mean Ratio: {test_mae_final/y_test.mean():.2%}")
print(f"Improvement over baseline: {(test_rmse_baseline - test_rmse_final)/test_rmse_baseline:.2%}")

# Save all model artifacts
import json
import pickle

# Save model
final_model.save('final_neural_network_model.h5')

# Save training history
with open('training_history.json', 'w') as f:
    json.dump(history_final.history, f)

# Save scaler
with open('feature_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Save feature names
with open('feature_names.txt', 'w') as f:
    f.write('\n'.join(X.columns.tolist()))

print("\nModel artifacts saved:")
print("  - final_neural_network_model.h5")
print("  - training_history.json")
print("  - feature_scaler.pkl")
print("  - feature_names.txt")